In [22]:
import math
import matplotlib.pyplot as plt

# Planetary gravities (m/s^2)
planetary_gravity = {
    "Mercury": 3.7,
    "Venus": 8.87,
    "Earth": 9.81,
    "Mars": 3.71,
    "Jupiter": 24.79,
    "Saturn": 10.44,
    "Uranus": 8.69,
    "Neptune": 11.15,
    "Pluto": 0.62,
}

# Simulation constants
v0 = 0  # Initial velocity (m/s)
y0 = 100  # Initial height (m)
dt = 0.01  # Time step (s)
coin_radius = 15
vertical_margin = 100  # Distance from the top of the screen

# Function to calculate the fall time and final velocity
def calculate_fall_time_and_velocity(g):
    fall_time = math.sqrt(2 * y0 / g)  # Time to fall from height y0
    final_velocity = math.sqrt(2 * g * y0)  # Final velocity using v = sqrt(2gh)
    return fall_time, final_velocity


In [ ]:
import pygame
import math
import sys

# Initialize Pygame
pygame.init()

# Screen dimensions
WIDTH, HEIGHT = 1800, 900
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Planetary Coin Drop Simulation")

# Enhanced color palette
COLORS = {
    'background': (15, 15, 35),
    'surface': (25, 25, 45),
    'primary': (64, 156, 255),
    'secondary': (255, 107, 107),
    'accent': (255, 193, 7),
    'success': (76, 175, 80),
    'white': (255, 255, 255),
    'light_gray': (200, 200, 200),
    'dark_gray': (100, 100, 100),
    'gold': (255, 215, 0),
    'coin_shadow': (0, 0, 0, 100),
    'button_hover': (84, 176, 255),
    'text_secondary': (160, 160, 180)
}

# Planet colors for visual distinction
PLANET_COLORS = {
    "Mercury": (169, 169, 169),
    "Venus": (255, 198, 73),
    "Earth": (100, 149, 237),
    "Mars": (205, 92, 92),
    "Jupiter": (255, 140, 0),
    "Saturn": (255, 215, 0),
    "Uranus": (64, 224, 208),
    "Neptune": (30, 144, 255),
    "Pluto": (139, 69, 19)
}

# Planetary gravities (m/s^2)
planetary_gravity = {
    "Mercury": 3.7,
    "Venus": 8.87,
    "Earth": 9.81,
    "Mars": 3.71,
    "Jupiter": 24.79,
    "Saturn": 10.44,
    "Uranus": 8.69,
    "Neptune": 11.15,
    "Pluto": 0.62,
}

# Simulation constants
v0 = 0  # Initial velocity (m/s)
y0 = 100  # Initial height (m)
dt = 0.016  # Time step (s) - 60 FPS
coin_radius = 20
vertical_margin = 120

# Fonts
font_large = pygame.font.SysFont('Arial', 28, bold=True)
font_medium = pygame.font.SysFont('Arial', 20)
font_small = pygame.font.SysFont('Arial', 16)
font_title = pygame.font.SysFont('Arial', 36, bold=True)

# Button class for better UX
class Button:
    def __init__(self, x, y, width, height, text, color=COLORS['primary'], hover_color=COLORS['button_hover'], text_color=COLORS['white']):
        self.rect = pygame.Rect(x, y, width, height)
        self.text = text
        self.color = color
        self.hover_color = hover_color
        self.text_color = text_color
        self.hovered = False
        self.clicked = False
        
    def handle_event(self, event):
        if event.type == pygame.MOUSEMOTION:
            self.hovered = self.rect.collidepoint(event.pos)
        elif event.type == pygame.MOUSEBUTTONDOWN:
            if self.rect.collidepoint(event.pos):
                self.clicked = True
                return True
        return False
    
    def draw(self, screen):
        # Button shadow
        shadow_rect = self.rect.copy()
        shadow_rect.x += 3
        shadow_rect.y += 3
        pygame.draw.rect(screen, (0, 0, 0, 50), shadow_rect, border_radius=8)
        
        # Button background
        color = self.hover_color if self.hovered else self.color
        pygame.draw.rect(screen, color, self.rect, border_radius=8)
        
        # Button border
        pygame.draw.rect(screen, COLORS['white'], self.rect, 2, border_radius=8)
        
        # Button text
        text_surface = font_medium.render(self.text, True, self.text_color)
        text_rect = text_surface.get_rect(center=self.rect.center)
        screen.blit(text_surface, text_rect)

# Create buttons
height_buttons = [
    Button(50, HEIGHT - 80, 120, 50, "50m", COLORS['success']),
    Button(190, HEIGHT - 80, 120, 50, "80m", COLORS['accent']),
    Button(330, HEIGHT - 80, 120, 50, "100m", COLORS['secondary']),
]

restart_button = Button(WIDTH - 200, 20, 150, 50, " Restart", COLORS['primary'])
info_button = Button(WIDTH - 370, 20, 150, 50, " Info", COLORS['surface'])

# Function to calculate the fall time and final velocity
def calculate_fall_time_and_velocity(g):
    fall_time = math.sqrt(2 * y0 / g)
    final_velocity = math.sqrt(2 * g * y0)
    return fall_time, final_velocity

# Function to reset the simulation
def reset_simulation():
    coins = []
    planets = list(planetary_gravity.items())
    
    for i, (planet, gravity) in enumerate(planets):
        fall_time, final_velocity = calculate_fall_time_and_velocity(gravity)
        scaling_factor = (HEIGHT - 2 * vertical_margin) / y0

        coins.append({
            "planet": planet,
            "x": 120 + i * 140,
            "y": vertical_margin,
            "v": v0,
            "g": gravity,
            "falling": True,
            "fall_time": 0,
            "true_fall_time": fall_time,
            "final_velocity": final_velocity,
            "scaling_factor": scaling_factor,
            "trail": [],  # For visual trail effect
        })
    return coins

# Function to draw coin with enhanced visuals
def draw_coin(screen, coin):
    x, y = int(coin["x"]), int(coin["y"])
    
    # Draw trail
    for i, (trail_x, trail_y) in enumerate(coin["trail"]):
        alpha = int(255 * (i / max(len(coin["trail"]), 1)) * 0.3)
        trail_surface = pygame.Surface((coin_radius, coin_radius), pygame.SRCALPHA)
        pygame.draw.circle(trail_surface, (*PLANET_COLORS[coin["planet"]], alpha), 
                         (coin_radius//2, coin_radius//2), coin_radius//3)
        screen.blit(trail_surface, (trail_x - coin_radius//2, trail_y - coin_radius//2))
    
    # Coin shadow
    pygame.draw.circle(screen, (0, 0, 0, 80), (x + 3, y + 3), coin_radius)
    
    # Coin body
    pygame.draw.circle(screen, PLANET_COLORS[coin["planet"]], (x, y), coin_radius)
    pygame.draw.circle(screen, COLORS['gold'], (x, y), coin_radius, 3)
    
    # Coin highlight
    pygame.draw.circle(screen, COLORS['white'], (x - 5, y - 5), coin_radius//4)

# Function to draw visualization graph
def draw_graph(surface, coins, graph_rect):
    # Graph background
    pygame.draw.rect(surface, (*COLORS['background'], 150), graph_rect, border_radius=5)
    pygame.draw.rect(surface, COLORS['primary'], graph_rect, 2, border_radius=5)
    
    # Graph title
    graph_title = font_medium.render("Fall Time Comparison", True, COLORS['white'])
    surface.blit(graph_title, (graph_rect.x + 10, graph_rect.y + 10))
    
    # Sort coins by fall time
    sorted_coins = sorted(coins, key=lambda x: x['true_fall_time'])
    
    if not sorted_coins:
        return
    
    # Graph dimensions
    graph_x = graph_rect.x + 20
    graph_y = graph_rect.y + 40
    graph_width = graph_rect.width - 40
    graph_height = graph_rect.height - 60
    
    # Find max fall time for scaling
    max_time = max(coin['true_fall_time'] for coin in sorted_coins)
    
    # Draw bars
    bar_width = graph_width // len(sorted_coins) - 5
    for i, coin in enumerate(sorted_coins):
        bar_height = int((coin['true_fall_time'] / max_time) * graph_height)
        bar_x = graph_x + i * (bar_width + 5)
        bar_y = graph_y + graph_height - bar_height
        
        # Bar background
        pygame.draw.rect(surface, PLANET_COLORS[coin['planet']], 
                        (bar_x, bar_y, bar_width, bar_height), border_radius=3)
        
        # Bar outline
        pygame.draw.rect(surface, COLORS['white'], 
                        (bar_x, bar_y, bar_width, bar_height), 2, border_radius=3)
        
        # Planet name (rotated)
        planet_text = font_small.render(coin['planet'][:3], True, COLORS['white'])
        text_rect = planet_text.get_rect()
        text_rect.center = (bar_x + bar_width//2, graph_y + graph_height + 15)
        surface.blit(planet_text, text_rect)
        
        # Time value
        time_text = font_small.render(f"{coin['true_fall_time']:.1f}s", True, COLORS['white'])
        time_rect = time_text.get_rect()
        time_rect.center = (bar_x + bar_width//2, bar_y - 10)
        surface.blit(time_text, time_rect)

# Function to draw info panel
def draw_info_panel(screen, coins):
    # Larger panel dimensions
    panel_rect = pygame.Rect(20, 80, 600, HEIGHT - 180)
    panel_surface = pygame.Surface(panel_rect.size, pygame.SRCALPHA)
    panel_surface.fill((*COLORS['surface'], 220))
    
    # Panel border with glow effect
    pygame.draw.rect(panel_surface, COLORS['primary'], panel_surface.get_rect(), 3, border_radius=15)
    
    # Title section
    title_rect = pygame.Rect(0, 0, panel_rect.width, 60)
    pygame.draw.rect(panel_surface, (*COLORS['primary'], 100), title_rect, border_radius=15)
    title = font_title.render(" Simulation Analytics", True, COLORS['white'])
    panel_surface.blit(title, (25, 15))
    
    # Current height display
    height_bg = pygame.Rect(20, 70, 200, 40)
    pygame.draw.rect(panel_surface, (*COLORS['accent'], 150), height_bg, border_radius=8)
    height_text = font_large.render(f"Drop Height: {y0}m", True, COLORS['white'])
    panel_surface.blit(height_text, (30, 80))
    
    # Graph section
    graph_rect = pygame.Rect(20, 120, 560, 200)
    draw_graph(panel_surface, coins, graph_rect)
    
    # Data table section
    table_y = 340
    table_header = font_medium.render("Detailed Results:", True, COLORS['accent'])
    panel_surface.blit(table_header, (50, table_y))
    
    # Table headers
    headers = ["Planet", "Gravity", "Fall Time", "Final Velocity", "Status"]
    header_positions = [20, 120, 220, 320, 450]
    
    for i, header in enumerate(headers):
        header_text = font_small.render(header, True, COLORS['light_gray'])
        panel_surface.blit(header_text, (header_positions[i], table_y + 30))
    
    # Draw separator line
    pygame.draw.line(panel_surface, COLORS['primary'], 
                    (20, table_y + 50), (580, table_y + 50), 2)
    
    # Planet data rows
    y_offset = table_y + 60
    for i, coin in enumerate(sorted(coins, key=lambda x: x['true_fall_time'])):
        if y_offset > panel_rect.height - 40:
            break
            
        # Alternating row background
        if i % 2 == 0:
            row_rect = pygame.Rect(15, y_offset - 5, 570, 25)
            pygame.draw.rect(panel_surface, (*COLORS['background'], 100), row_rect, border_radius=5)
        
        color = PLANET_COLORS[coin["planet"]]
        
        # Planet name with colored indicator
        pygame.draw.circle(panel_surface, color, (30, y_offset + 8), 6)
        planet_text = font_small.render(coin["planet"], True, COLORS['white'])
        panel_surface.blit(planet_text, (45, y_offset))
        
        # Gravity
        gravity_text = font_small.render(f"{coin['g']:.2f} m/s²", True, COLORS['text_secondary'])
        panel_surface.blit(gravity_text, (header_positions[1], y_offset))
        
        # Fall time with progress bar
        time_text = font_small.render(f"{coin['fall_time']:.2f}s", True, COLORS['white'])
        panel_surface.blit(time_text, (header_positions[2], y_offset))
        
        # Final velocity
        velocity_text = font_small.render(f"{coin['v']:.1f} m/s", True, COLORS['text_secondary'])
        panel_surface.blit(velocity_text, (header_positions[3], y_offset))
        
        # Status indicator
        status_color = COLORS['success'] if not coin['falling'] else COLORS['accent']
        status_text = "Complete" if not coin['falling'] else "Falling"
        status_surface = font_small.render(status_text, True, status_color)
        panel_surface.blit(status_surface, (header_positions[4], y_offset))
        
        y_offset += 30
    
    # Physics formula section
    if y_offset < panel_rect.height - 80:
        formula_y = panel_rect.height - 70
        formula_bg = pygame.Rect(20, formula_y - 10, 560, 60)
        pygame.draw.rect(panel_surface, (*COLORS['background'], 180), formula_bg, border_radius=8)
        
        formula_title = font_medium.render("Physics Formula:", True, COLORS['accent'])
        panel_surface.blit(formula_title, (30, formula_y))
        
        formula_text = font_small.render("t = √(2h/g)  |  v = √(2gh)  |  where h = height, g = gravity", True, COLORS['light_gray'])
        panel_surface.blit(formula_text, (30, formula_y + 25))
    
    screen.blit(panel_surface, panel_rect)

# Function to draw background grid
def draw_background(screen):
    screen.fill(COLORS['background'])
    
    # Grid lines
    for x in range(0, WIDTH, 50):
        pygame.draw.line(screen, (*COLORS['surface'], 100), (x, 0), (x, HEIGHT))
    for y in range(0, HEIGHT, 50):
        pygame.draw.line(screen, (*COLORS['surface'], 100), (0, y), (WIDTH, y))
    
    # Title
    title = font_title.render(" Planetary Coin Drop Simulation", True, COLORS['white'])
    title_rect = title.get_rect(center=(WIDTH//2, 40))
    
    # Title background
    title_bg = pygame.Rect(title_rect.x - 20, title_rect.y - 10, title_rect.width + 40, title_rect.height + 20)
    pygame.draw.rect(screen, (*COLORS['surface'], 150), title_bg, border_radius=10)
    
    screen.blit(title, title_rect)

# Initialize simulation
coins = reset_simulation()
show_info = False

# Main simulation loop
running = True
clock = pygame.time.Clock()

while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        
        # Handle button events
        if restart_button.handle_event(event):
            coins = reset_simulation()
        
        if info_button.handle_event(event):
            show_info = not show_info
        
        # Handle height buttons
        for i, button in enumerate(height_buttons):
            if button.handle_event(event):
                heights = [50, 80, 100]
                y0 = heights[i]
                coins = reset_simulation()

    # Draw background
    draw_background(screen)
    
    # Update and draw coins
    for coin in coins:
        if coin["falling"]:
            coin["fall_time"] += dt
            coin["v"] = min(coin["v"] + coin["g"] * dt, coin["final_velocity"])
            coin["y"] += coin["v"] * dt * coin["scaling_factor"]
            
            # Add to trail
            coin["trail"].append((coin["x"], coin["y"]))
            if len(coin["trail"]) > 10:
                coin["trail"].pop(0)
            
            # Check if reached ground
            if coin["fall_time"] >= coin["true_fall_time"]:
                coin["falling"] = False
                coin["fall_time"] = coin["true_fall_time"]
                coin["v"] = coin["final_velocity"]
        
        # Draw coin
        draw_coin(screen, coin)
        
        # Draw coin info
        info_y = coin["y"] - 60
        planet_text = font_medium.render(coin['planet'], True, PLANET_COLORS[coin["planet"]])
        time_text = font_small.render(f"T: {coin['fall_time']:.2f}s", True, COLORS['white'])
        velocity_text = font_small.render(f"V: {coin['v']:.1f} m/s", True, COLORS['white'])
        
        # Info background
        info_rect = pygame.Rect(coin["x"] - 50, info_y - 10, 100, 50)
        info_surface = pygame.Surface(info_rect.size, pygame.SRCALPHA)
        info_surface.fill((*COLORS['surface'], 180))
        pygame.draw.rect(info_surface, COLORS['primary'], info_surface.get_rect(), 1, border_radius=5)
        screen.blit(info_surface, info_rect)
        
        # Info text
        screen.blit(planet_text, (coin["x"] - planet_text.get_width()//2, info_y))
        screen.blit(time_text, (coin["x"] - time_text.get_width()//2, info_y + 15))
        screen.blit(velocity_text, (coin["x"] - velocity_text.get_width()//2, info_y + 30))

    # Draw ground line
    ground_y = HEIGHT - vertical_margin
    pygame.draw.line(screen, COLORS['light_gray'], (0, ground_y), (WIDTH, ground_y), 3)
    ground_text = font_small.render("Ground Level", True, COLORS['light_gray'])
    screen.blit(ground_text, (WIDTH - 120, ground_y + 10))

    # Draw buttons
    for button in height_buttons:
        button.draw(screen)
    
    restart_button.draw(screen)
    info_button.draw(screen)
    
    # Draw info panel if enabled
    if show_info:
        draw_info_panel(screen, coins)
    
    # Draw instructions
    instructions = [
        "Click height buttons to change drop height",
        "Click 'Info' to toggle detailed panel",
        "Watch coins fall at different rates based on planetary gravity!"
    ]
    
    for i, instruction in enumerate(instructions):
        text = font_small.render(instruction, True, COLORS['text_secondary'])
        screen.blit(text, (50, HEIGHT - 140 + i * 200))

    # Update display
    pygame.display.flip()
    clock.tick(60)

pygame.quit()
sys.exit()

SystemExit: 

In [ ]:
# Function to simulate and plot results for different heights
def simulate_and_plot_results():
    heights = [50, 80, 100]  # Heights to simulate

    # Simulate results for each height and store the fall times and velocities
    for height in heights:
        global y0
        y0 = height  # Set the initial height
        coins = reset_simulation()  # Reset simulation with the specified height

        # Prepare data for plotting
        fall_times = []
        final_velocities = []
        accelerations = []
        planets = []

        for coin in coins:
            fall_times.append(coin['true_fall_time'])
            final_velocities.append(math.sqrt(2 * coin['g'] * height))  # Calculate final velocity
            accelerations.append(coin['g'])  # The acceleration is the gravitational pull for each planet
            planets.append(coin['planet'])

        # Create line plots for each height
        fig, ax1 = plt.subplots(figsize=(8, 6))

        # Plot for Fall Time and Final Velocity
        ax1.plot(planets, fall_times, marker='o', color='blue', label="Fall Time (s)")
        ax1.plot(planets, final_velocities, marker='s', color='green', label="Final Velocity (m/s)")
        ax1.set_xlabel("Planet")
        ax1.set_ylabel("Time (s) / Velocity (m/s)")
        ax1.set_title(f"Fall Time and Final Velocity for Height: {height}m")
        ax1.set_xticks(range(len(planets)))
        ax1.set_xticklabels(planets)

        # Create a second y-axis for the acceleration
        ax2 = ax1.twinx()
        ax2.plot(planets, accelerations, marker='d', color='red', label="Acceleration (m/s²)", linestyle='--')
        ax2.set_ylabel("Acceleration (m/s²)")

        # Display both plots for the current height
        ax1.legend(loc='upper left')
        ax2.legend(loc='upper right')
        plt.tight_layout()
        plt.show()

# Run the plotting function
simulate_and_plot_results()
